In [13]:
import requests
import io
import pandas
import numpy as np
#import matplotlib.pyplot as plt
from subprocess import Popen
import platform
import os
import shutil
from getpass import getpass
import netrc
from base64 import b64encode
from datetime import datetime

In [14]:
from tsgettoolbox import tsgettoolbox as tsget
from wdmtoolbox import wdmtoolbox as wdm


# Establish files required for API Access

In [24]:
# Set homeDir to current working directory
homeDir = os.path.expanduser("~/")
#homeDir = os.getcwd() + os.sep
# Create .urs_cookies and .dodsrc files in current directory
with open(homeDir + '.urs_cookies', 'w') as file:
    file.write('')
with open(homeDir + '.dodsrc', 'w') as file:
    file.write('HTTP.COOKIEJAR={}.urs_cookies\n'.format(homeDir))
    file.write('HTTP.NETRC={}.netrc'.format(homeDir))

print('Saved .urs_cookies and .dodsrc to:', homeDir)

# No need to copy .dodsrc since it's already in the working directory

Saved .urs_cookies and .dodsrc to: C:\Users\KVENABLE/


## Login with your Earthdata account. When completed your netrc file will be developed 

In [25]:
urs = 'urs.earthdata.nasa.gov'    # Earthdata URL to call for authentication
prompts = ['Enter NASA Earthdata Login Username \n(or create an account at urs.earthdata.nasa.gov): ',
           'Enter NASA Earthdata Login Password: ']

with open(homeDir + '.netrc', 'w') as file:
    file.write('machine {} login {} password {}'.format(urs, getpass(prompt=prompts[0]), getpass(prompt=prompts[1])))
    file.close()

print('Saved .netrc to:', homeDir)

# Set appropriate permissions for Linux/macOS
if platform.system() != "Windows":
    Popen('chmod og-rw ~/.netrc', shell=True)

Enter NASA Earthdata Login Username 
(or create an account at urs.earthdata.nasa.gov):  ········
Enter NASA Earthdata Login Password:  ········


Saved .netrc to: C:\Users\KVENABLE/


## Authenticate Credentials from login to generate token 

In [26]:
# Earthdata Login URL for obtaining the token, and creating one if it doesn't exist
url = 'https://urs.earthdata.nasa.gov/api/users/find_or_create_token'

# Earthdata Login credential prompts
prompts = ['Enter NASA Earthdata Login Username \n(or create an account at urs.earthdata.nasa.gov): ',
           'Enter NASA Earthdata Login Password: ']

# Get credentials from user input
auth = netrc.netrc()
#auth = netrc.netrc(file=homeDir + '.netrc')
username, _, password = auth.authenticators('urs.earthdata.nasa.gov')

# Encode credentials using Base64
credentials = b64encode(f"{username}:{password}".encode('utf-8')).decode('utf-8')

# Headers with the Basic Authorization
headers = {
    'Authorization': f'Basic {credentials}'
}

# Make the POST request to get the token
response = requests.post(url, headers=headers)

# Check if the request was successful
if response.status_code == 200:
    # Parse the response JSON to get the token
    token_info = response.json()
    token = token_info.get("access_token")
    print("Token retrieved successfully")

    # Define the path for the .edl_token file in the home directory
    token_file_path = homeDir + ".edl_token"

    # Write the token to the .edl_token file
    with open(token_file_path, 'w') as token_file:
        token_file.write(token)

    print(f"Token saved to {token_file_path}")

else:
    print("Failed to retrieve token:", response.text)

Token retrieved successfully
Token saved to C:\Users\KVENABLE/.edl_token


In [27]:
def convertunitforHSPF(constituent, df, LDAS_var):
    '''This function is for unit conversion'''
    if constituent == "ATEMP": df = (df-273)
    #From K to degC
    if constituent == "PRECIP" and 'GLDAS' in LDAS_var:
        df = df * 3600
        #From kg/m^2/s to mm/hour
        #Assuming that 1 kg/m^2 is close to 1 mm
        df=df*3
            #GLDAS is 3 hourly data. THis changes values from
            #mm/hour to total precip for each times step of 3 hours.
            #There might be a better way to accomplish this task
    elif constituent == "PRECIP" and 'NLDAS' in LDAS_var:
        df=df
        #No change needed for NLDAS            
   
    return df


In [28]:
StationList=[['Seattle',47.629605, -122.348941,-8],
            ['Allahabad', 25.43,81.84,5.5],
            ['Asmara',15.33, 38.92,3]]


In [29]:
Constituent=['ATEMP','PRECIP']
#Please refer to the GLDAS2 documentation below for more constituents.
#https://hydro1.gesdisc.eosdis.nasa.gov/data/GLDAS/README_GLDAS2.pdf

GLDAS_ConstituentDetails={
                    "PRECIP":["Precipitation","mm",
                    "GLDAS2:GLDAS_NOAH025_3H_2_1_Rainf_f_tavg","kg/m^2/s"],
                    "ATEMP":["Air Temperature","Fahrenheit",
                    "GLDAS2:GLDAS_NOAH025_3H_2_1_Tair_f_inst","K"]
                    }

NLDAS_ConstituentDetails={
                    "PRECIP":["Precipitation","mm",
                    "NLDAS:NLDAS_FORA0125_H_2_0_Rainf","mm"],
                    "ATEMP":["Air Temperature","Fahrenheit",
                    "NLDAS:NLDAS_FORA0125_H_2_0_Tair","K"]
                    }

#If you add more constituents, you will need to expland this dict


In [30]:

UStop = 49.3457868 # north lat
USleft = -124.7844079 # west long
USright = -66.9513812 # east long
USbottom =  24.7433195 # south lat
#US Lat and long coordinates


In [10]:
WDMFileName='MetData_030426.wdm'
wdm.createnewwdm(WDMFileName, overwrite=True)
index = 1
from datetime import datetime
with open("MetLog.txt", 'w') as Logfile:
    Logfile.write("Started Downloading the data at "
                + datetime.isoformat(datetime.now()) + " and saving in "
                + WDMFileName + "\n")
    for station in StationList:
        # Going through Each Station in the list
        TimeZoneAdjustment = station[3]
        Logfile.write("Station: " + station[0] + ", Latitude: " + str(station[1])
                        + ", Longitude: " + str(station[2])
                        + ", TimeZoneAdjustment: " + str(TimeZoneAdjustment)
                        + "\n")

        for const in Constituent:
            #Going through each constituent
            if station[1]<UStop and station[1]>USbottom and \
                station[2]>USleft and station[2]<USright:
                LDAS_variable = NLDAS_ConstituentDetails[const][2]  
                TimeStep=1  
                print(f'{station[0]} is in contiguous USA')
            else:
                LDAS_variable = GLDAS_ConstituentDetails[const][2]
                TimeStep=3
                print(f'{station} is outside contiguous USA')
            print(LDAS_variable)
            stationID = station
            print("Downloading " + const + " data for grid: " + station[0])
            df = tsget.ldas(lat=station[1], lon=station[2],
                               variables=LDAS_variable,
                               startDate="2018-12-31",
                               endDate="2019-1-31")
            column_name = df.columns[0]
            df = df[column_name]
            df.dropna()
            df = convertunitforHSPF(const,df, LDAS_variable)
            wdm.createnewdsn(WDMFileName, index,
                                constituent=const,
                                scenario="OBSERVED",location=station[0][0:8],
                                tcode=3, statid=station[0], tsstep=TimeStep,
                                description=LDAS_variable)
            #Creating an empty dataset in WDM File
           
            wdm.csvtowdm(WDMFileName, index, input_ts=df)
            #saving the data in the DSN created in previous line
           
            Logfile.write("Constituent: " + const + ", Column Name:"
            + column_name + ", DSN: " + str(index) + "\n")
            index += 1



Seattle is in contiguous USA
NLDAS:NLDAS_FORA0125_H_2_0_Tair


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\KVENABLE\\.netrc'

In [11]:
#Testing after moving to that directory 
WDMFileName='MetData_030426.wdm'
wdm.createnewwdm(WDMFileName, overwrite=True)
index = 1
from datetime import datetime
with open("MetLog.txt", 'w') as Logfile:
    Logfile.write("Started Downloading the data at "
                + datetime.isoformat(datetime.now()) + " and saving in "
                + WDMFileName + "\n")
    for station in StationList:
        # Going through Each Station in the list
        TimeZoneAdjustment = station[3]
        Logfile.write("Station: " + station[0] + ", Latitude: " + str(station[1])
                        + ", Longitude: " + str(station[2])
                        + ", TimeZoneAdjustment: " + str(TimeZoneAdjustment)
                        + "\n")

        for const in Constituent:
            #Going through each constituent
            if station[1]<UStop and station[1]>USbottom and \
                station[2]>USleft and station[2]<USright:
                LDAS_variable = NLDAS_ConstituentDetails[const][2]  
                TimeStep=1  
                print(f'{station[0]} is in contiguous USA')
            else:
                LDAS_variable = GLDAS_ConstituentDetails[const][2]
                TimeStep=3
                print(f'{station} is outside contiguous USA')
            print(LDAS_variable)
            stationID = station
            print("Downloading " + const + " data for grid: " + station[0])
            df = tsget.ldas(lat=station[1], lon=station[2],
                               variables=LDAS_variable,
                               startDate="2018-12-31",
                               endDate="2019-1-31")
            column_name = df.columns[0]
            df = df[column_name]
            df.dropna()
            df = convertunitforHSPF(const,df, LDAS_variable)
            wdm.createnewdsn(WDMFileName, index,
                                constituent=const,
                                scenario="OBSERVED",location=station[0][0:8],
                                tcode=3, statid=station[0], tsstep=TimeStep,
                                description=LDAS_variable)
            #Creating an empty dataset in WDM File
           
            wdm.csvtowdm(WDMFileName, index, input_ts=df)
            #saving the data in the DSN created in previous line
           
            Logfile.write("Constituent: " + const + ", Column Name:"
            + column_name + ", DSN: " + str(index) + "\n")
            index += 1

Seattle is in contiguous USA
NLDAS:NLDAS_FORA0125_H_2_0_Tair
Seattle is in contiguous USA
NLDAS:NLDAS_FORA0125_H_2_0_Rainf
['Allahabad', 25.43, 81.84, 5.5] is outside contiguous USA
GLDAS2:GLDAS_NOAH025_3H_2_1_Tair_f_inst
['Allahabad', 25.43, 81.84, 5.5] is outside contiguous USA
GLDAS2:GLDAS_NOAH025_3H_2_1_Rainf_f_tavg
['Asmara', 15.33, 38.92, 3] is outside contiguous USA
GLDAS2:GLDAS_NOAH025_3H_2_1_Tair_f_inst
['Asmara', 15.33, 38.92, 3] is outside contiguous USA
GLDAS2:GLDAS_NOAH025_3H_2_1_Rainf_f_tavg


In [31]:
#Testing after adding file creation and placing into default directory
WDMFileName='MetData_030426b.wdm'
wdm.createnewwdm(WDMFileName, overwrite=True)
index = 1
from datetime import datetime
with open("MetLogb.txt", 'w') as Logfile:
    Logfile.write("Started Downloading the data at "
                + datetime.isoformat(datetime.now()) + " and saving in "
                + WDMFileName + "\n")
    for station in StationList:
        # Going through Each Station in the list
        TimeZoneAdjustment = station[3]
        Logfile.write("Station: " + station[0] + ", Latitude: " + str(station[1])
                        + ", Longitude: " + str(station[2])
                        + ", TimeZoneAdjustment: " + str(TimeZoneAdjustment)
                        + "\n")

        for const in Constituent:
            #Going through each constituent
            if station[1]<UStop and station[1]>USbottom and \
                station[2]>USleft and station[2]<USright:
                LDAS_variable = NLDAS_ConstituentDetails[const][2]  
                TimeStep=1  
                print(f'{station[0]} is in contiguous USA')
            else:
                LDAS_variable = GLDAS_ConstituentDetails[const][2]
                TimeStep=3
                print(f'{station} is outside contiguous USA')
            print(LDAS_variable)
            stationID = station
            print("Downloading " + const + " data for grid: " + station[0])
            df = tsget.ldas(lat=station[1], lon=station[2],
                               variables=LDAS_variable,
                               startDate="2018-12-31",
                               endDate="2019-1-31")
            column_name = df.columns[0]
            df = df[column_name]
            df.dropna()
            df = convertunitforHSPF(const,df, LDAS_variable)
            wdm.createnewdsn(WDMFileName, index,
                                constituent=const,
                                scenario="OBSERVED",location=station[0][0:8],
                                tcode=3, statid=station[0], tsstep=TimeStep,
                                description=LDAS_variable)
            #Creating an empty dataset in WDM File
           
            wdm.csvtowdm(WDMFileName, index, input_ts=df)
            #saving the data in the DSN created in previous line
           
            Logfile.write("Constituent: " + const + ", Column Name:"
            + column_name + ", DSN: " + str(index) + "\n")
            index += 1

Seattle is in contiguous USA
NLDAS:NLDAS_FORA0125_H_2_0_Tair
Seattle is in contiguous USA
NLDAS:NLDAS_FORA0125_H_2_0_Rainf
['Allahabad', 25.43, 81.84, 5.5] is outside contiguous USA
GLDAS2:GLDAS_NOAH025_3H_2_1_Tair_f_inst
['Allahabad', 25.43, 81.84, 5.5] is outside contiguous USA
GLDAS2:GLDAS_NOAH025_3H_2_1_Rainf_f_tavg
['Asmara', 15.33, 38.92, 3] is outside contiguous USA
GLDAS2:GLDAS_NOAH025_3H_2_1_Tair_f_inst
['Asmara', 15.33, 38.92, 3] is outside contiguous USA
GLDAS2:GLDAS_NOAH025_3H_2_1_Rainf_f_tavg
